# Generative Adversarial Networks

**Generative Adversarial Networks (GANs)**, introduced by Ian Goodfellow and colleagues in 2014, are a class of deep learning models that learn to generate realistic data by pitting two neural networks against each other in a competitive game. One network — the **generator** — creates synthetic samples from random noise. The other — the **discriminator** — tries to distinguish real data from fakes. As training progresses, the generator improves until its outputs are indistinguishable from real examples.

GANs revolutionized image generation and remain foundational to modern generative AI. They power face synthesis, style transfer, data augmentation, and many techniques that diffusion models and large language models later extended. Understanding GANs clarifies how adversarial training works, why generative models are hard to train, and how architectural innovations address those challenges.

This notebook covers how GANs work, their core architecture, conditional variants, notable extensions (StyleGAN and CycleGAN), the training process including loss functions and training loops, and real-world applications. The explanations are self-contained and build on concepts from deep learning and generative AI fundamentals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. How GANs Work

A GAN frames generation as a **two-player minimax game**. The generator $G$ and discriminator $D$ have opposing goals:

- **Generator** — maps random noise $z$ (sampled from a simple distribution such as a standard normal) into synthetic data $G(z)$ that resembles real samples.
- **Discriminator** — receives either a real sample $x$ or a fake sample $G(z)$ and outputs a probability that the input is real.

The generator wants to fool the discriminator. The discriminator wants to correctly label real and fake data. Neither can succeed permanently — as the discriminator improves, the generator must produce more convincing fakes; as the generator improves, the discriminator must find subtler differences.

At equilibrium, the generator has learned the data distribution well enough that the discriminator can do no better than random guessing ($D(x) = 0.5$ for both real and generated samples).

The original GAN objective is expressed as a minimax value function:

$$\min_G \max_D \; V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}}\big[\log D(x)\big] + \mathbb{E}_{z \sim p_z}\big[\log(1 - D(G(z)))\big]$$

The discriminator maximizes $V$ by assigning high scores to real data and low scores to fakes. The generator minimizes $V$ by making $D(G(z))$ approach 1 — convincing the discriminator that fakes are real.

```
  Noise z                    Real data x
     │                           │
     ▼                           ▼
 ┌──────────┐              ┌──────────┐
 │Generator │              │          │
 │   G(z)   │              │    x     │
 └────┬─────┘              └────┬─────┘
      │                         │
      └──────────┬──────────────┘
                 ▼
         ┌──────────────┐
         │ Discriminator│  →  P(real)
         │      D(·)    │
         └──────────────┘
```

Unlike autoencoders, which learn an explicit encoding-decoding bottleneck, GANs learn generation **implicitly** through adversarial feedback. There is no reconstruction loss — only the discriminator's judgment guides the generator. This makes GANs flexible but also notoriously difficult to stabilize during training.

---

## 2. GAN Architecture: Generator and Discriminator

Every GAN consists of two neural networks with complementary roles. Their architectures depend on the data modality, but the conceptual split is universal.

### 2.1 The Generator

The generator $G: \mathbb{R}^{d_z} \rightarrow \mathbb{R}^{d_x}$ transforms a low-dimensional noise vector $z \in \mathbb{R}^{d_z}$ into a high-dimensional sample (an image, audio clip, or other data). For image generation, the generator typically uses **transposed convolutions** (also called deconvolutions) to upsample from a small spatial feature map to full resolution.

A typical image generator structure:

```
z (100-dim noise) → Dense → Reshape (4×4×512)
  → Transposed Conv (8×8×256) → BatchNorm → ReLU
  → Transposed Conv (16×16×128) → BatchNorm → ReLU
  → Transposed Conv (32×32×64)  → BatchNorm → ReLU
  → Transposed Conv (64×64×3)   → Tanh → Generated image
```

The final activation is often **Tanh** (output in $[-1, 1]$) when training images are normalized to that range, or **Sigmoid** for $[0, 1]$ outputs.

### 2.2 The Discriminator

The discriminator $D: \mathbb{R}^{d_x} \rightarrow [0, 1]$ is a **binary classifier**. For images, it mirrors a standard convolutional network: strided convolutions progressively downsample spatial dimensions until a single scalar probability is produced via a sigmoid activation.

```
Image (64×64×3)
  → Conv (32×32×64)  → LeakyReLU
  → Conv (16×16×128) → BatchNorm → LeakyReLU
  → Conv (8×8×256)   → BatchNorm → LeakyReLU
  → Conv (4×4×512)   → BatchNorm → LeakyReLU
  → Flatten → Dense(1) → Sigmoid → P(real)
```

**LeakyReLU** is preferred over ReLU in the discriminator because it allows small negative gradients, which helps gradients flow when the discriminator is confident.

### 2.3 Design Principles

| Component | Role | Common Choices |
|-----------|------|----------------|
| **Generator** | Upsample noise to data | Transposed convolutions, batch normalization, ReLU |
| **Discriminator** | Classify real vs fake | Strided convolutions, LeakyReLU, dropout |
| **Noise input** | Source of randomness | $z \sim \mathcal{N}(0, I)$ or uniform distribution |
| **Output activation** | Match data range | Tanh or Sigmoid |

The generator and discriminator must be **balanced in capacity**. If the discriminator is too strong, it rejects all fakes immediately and the generator receives vanishing gradients. If the generator is too strong too early, the discriminator cannot provide useful learning signals. This balance is one of the central practical challenges in GAN training.

In [ ]:
# Toy 1D GAN: generator maps noise to a scalar; discriminator classifies real vs fake

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

# Real data: samples from N(2.0, 0.3)
real_data = np.random.normal(2.0, 0.3, 500)

# Generator: z -> mu + sigma * z  (learnable location and scale)
g_mu, g_sigma = 0.0, 1.0

# Discriminator: x -> sigmoid(w * x + b)
d_w, d_b = 0.5, -0.5

def generate(n=200):
    z = np.random.randn(n)
    return g_mu + g_sigma * z

def discriminate(x):
    return sigmoid(d_w * x + d_b)

fake_data = generate(500)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(real_data, bins=30, alpha=0.6, label="Real (N(2, 0.3))", density=True)
axes[0].hist(fake_data, bins=30, alpha=0.6, label="Fake G(z)", density=True)
axes[0].set_title("Generator Output vs Real Data")
axes[0].set_xlabel("Value")
axes[0].legend()

x_grid = np.linspace(-1, 4, 200)
axes[1].plot(x_grid, discriminate(x_grid), color="crimson", lw=2)
axes[1].axhline(0.5, color="gray", ls="--", alpha=0.7, label="Decision boundary (0.5)")
axes[1].set_title("Discriminator Output D(x)")
axes[1].set_xlabel("x")
axes[1].set_ylabel("P(real)")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Mean of real data: {real_data.mean():.3f}")
print(f"Mean of fake data: {fake_data.mean():.3f}")
print(f"Discriminator on real (avg): {discriminate(real_data).mean():.3f}")
print(f"Discriminator on fake (avg): {discriminate(fake_data).mean():.3f}")

---

## 3. Conditional GANs

A standard GAN generates samples from a fixed, unlabeled distribution — it might produce faces, but you cannot control whose face or what expression. A **Conditional GAN (cGAN)** adds **side information** $c$ (a class label, text embedding, or other conditioning signal) to both the generator and discriminator, enabling **controlled generation**.

The conditional objective becomes:

$$\min_G \max_D \; V(D, G) = \mathbb{E}_{x,c}\big[\log D(x \mid c)\big] + \mathbb{E}_{z,c}\big[\log(1 - D(G(z \mid c) \mid c))\big]$$

Both networks receive the condition $c$:

- **Generator** — takes $(z, c)$ and produces $G(z, c)$. For example, $c$ might be a digit label (0–9) and $G$ produces an image of that digit.
- **Discriminator** — takes $(x, c)$ and judges whether the pair is a real image with the correct label or a generated fake.

```
  z (noise) ──┐
              ├──► Generator G(z, c) ──► fake image
  c (label) ──┘                              │
                                             ▼
  real image ──┐                      ┌─────────────┐
               ├──► Discriminator ──►  P(real | x, c)
  c (label) ───┘                      └─────────────┘
```

Conditioning is typically implemented by **concatenating** the label (as a one-hot vector or embedding) with the noise input for the generator, and concatenating it with the image features for the discriminator.

### 3.1 Why Conditioning Matters

Conditional GANs bridge the gap between unconditional generation and practical applications:

- **Class-conditional generation** — produce images of a specific category (e.g., "generate a handwritten 7").
- **Text-to-image** — later architectures like StackGAN and DALL-E extend this idea with text encoders as conditions.
- **Image-to-image translation** — Pix2Pix uses paired conditioning (input image + target) for supervised translation.
- **Data augmentation** — generate labeled synthetic training data for underrepresented classes.

The key insight is that $c$ provides a **handle** on the latent space. Instead of wandering randomly through all possible outputs, the user steers generation toward a desired outcome.

---

## 4. StyleGAN and CycleGAN

After the original GAN, researchers proposed many architectural innovations. Two of the most influential are **StyleGAN**, which achieves remarkable control over image quality and attributes, and **CycleGAN**, which enables unpaired image-to-image translation.

### 4.1 StyleGAN

**StyleGAN** (Karras et al., 2019) redesigned the generator to control image synthesis at multiple scales through a **style-based architecture**. Instead of feeding noise only at the input layer, StyleGAN injects learned **style vectors** at each convolutional layer via **Adaptive Instance Normalization (AdaIN)**.

Key innovations:

- **Mapping network** — transforms the input noise $z$ into an intermediate latent space $w$, which is more disentangled (individual dimensions control distinct visual attributes).
- **Style injection** — each layer receives a style vector that controls coarse features (pose, face shape) at early layers and fine details (hair texture, skin pores) at later layers.
- **Progressive growing** — early versions trained on low-resolution images first, then progressively added layers for higher resolution. StyleGAN2 replaced this with a more stable single-pass design.
- **Noise injection** — per-layer stochastic noise adds fine-grained detail and natural variation.

```
z → Mapping Network → w → Style vectors (one per layer)
                              ↓
Constant 4×4 ──► Conv + AdaIN(w₁) ──► Conv + AdaIN(w₂) ──► ... ──► 1024×1024 image
```

StyleGAN produces photorealistic faces and objects with unprecedented quality. Its latent space supports **semantic manipulation** — interpolating between two $w$ vectors morphs one face into another, and shifting individual dimensions changes attributes like age, smile, or hair color.

### 4.2 CycleGAN

**CycleGAN** (Zhu et al., 2017) solves **unpaired image-to-image translation** — converting images from domain A to domain B without requiring matched training pairs. For example, turning horses into zebras or summer landscapes into winter scenes.

CycleGAN uses **two generators** and **two discriminators**:

- $G_{A \rightarrow B}$: maps domain A images to domain B style.
- $G_{B \rightarrow A}$: maps domain B images back to domain A style.
- $D_A$: discriminates real vs fake in domain A.
- $D_B$: discriminates real vs fake in domain B.

The critical addition is **cycle consistency loss**:

$$\mathcal{L}_{\text{cycle}} = \mathbb{E}_{x \sim A}\big[\| G_{B \rightarrow A}(G_{A \rightarrow B}(x)) - x \|_1\big] + \mathbb{E}_{y \sim B}\big[\| G_{A \rightarrow B}(G_{B \rightarrow A}(y)) - y \|_1\big]$$

This ensures that translating an image to the other domain and back recovers the original — preventing the generators from producing valid but meaningless outputs. Without paired data, cycle consistency provides the structural constraint that makes unpaired translation possible.

| Model | Problem Solved | Key Mechanism |
|-------|---------------|---------------|
| **StyleGAN** | High-quality controlled image synthesis | Style vectors, AdaIN, mapping network |
| **CycleGAN** | Unpaired domain translation | Dual generators, cycle consistency loss |
| **Pix2Pix** | Paired image-to-image translation | Conditional GAN with paired supervision |
| **DCGAN** | Stable training on simple images | Convolutional generator/discriminator design rules |

---

## 5. Training a GAN

GAN training is an iterative adversarial process. Unlike standard supervised learning with a single loss minimized by one network, GANs alternate between training the discriminator and the generator, each with its own objective.

### 5.1 Loss Functions

**Discriminator loss** — maximize the probability of correctly classifying real and fake samples. In practice, this is minimized as:

$$\mathcal{L}_D = -\mathbb{E}_{x \sim p_{\text{data}}}\big[\log D(x)\big] - \mathbb{E}_{z \sim p_z}\big[\log(1 - D(G(z)))\big]$$

**Generator loss** — the original formulation minimizes $\mathbb{E}_z[\log(1 - D(G(z)))]$, but this causes **vanishing gradients** when the discriminator easily rejects fakes (since $\log(1 - D(G(z))) \approx 0$ when $D(G(z)) \approx 0$).

A widely used alternative **relabels the generator's goal** — instead of minimizing the probability of being caught, the generator maximizes the probability of fooling the discriminator:

$$\mathcal{L}_G = -\mathbb{E}_{z \sim p_z}\big[\log D(G(z))\big]$$

This provides stronger gradients early in training when fakes are easily detected.

### 5.2 The Training Loop

A standard GAN training loop alternates discriminator and generator updates:

```
for epoch in range(num_epochs):
    for batch in dataloader:
        # --- Train Discriminator ---
        real_images = batch
        z = sample_noise(batch_size)
        fake_images = generator(z)

        d_loss_real = -log(D(real_images))
        d_loss_fake = -log(1 - D(fake_images.detach()))
        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        d_optimizer.step()

        # --- Train Generator ---
        z = sample_noise(batch_size)
        fake_images = generator(z)
        g_loss = -log(D(fake_images))          # fool the discriminator
        g_loss.backward()
        g_optimizer.step()
```

Note that fake images are **detached** during discriminator training so gradients do not flow into the generator when updating $D$.

### 5.3 Training Challenges

GAN training is notoriously unstable. Common issues include:

**Mode collapse** — the generator produces a limited variety of outputs (e.g., the same face repeatedly) because those outputs consistently fool the discriminator. The generator finds a local optimum and stops exploring.

**Non-convergence** — loss oscillates rather than decreasing. Unlike supervised learning, GAN losses do not reliably indicate training quality — a discriminator loss near zero might mean a strong discriminator or a collapsed generator.

**Vanishing gradients** — when the discriminator is too confident, the generator receives weak gradient signals and stops improving.

### 5.4 Stabilization Techniques

Practitioners use several techniques to improve training stability:

- **Label smoothing** — use 0.9 instead of 1.0 for real labels, preventing the discriminator from becoming overconfident.
- **Spectral normalization** — constrains the Lipschitz constant of the discriminator, limiting how fast it can improve.
- **Wasserstein GAN (WGAN)** — replaces the Jensen-Shannon divergence with Wasserstein distance and uses a critic instead of a discriminator, providing more meaningful loss curves.
- **Two Time-Scale Update Rule (TTUR)** — use different learning rates for generator and discriminator.
- **Mini-batch discrimination** — let the discriminator compare samples within a batch, discouraging mode collapse.
- **Progressive training** — start at low resolution and gradually increase (used in ProGAN and early StyleGAN).

In [ ]:
# Simulated GAN training curves and a minimal alternating update loop

epochs = 80
t = np.arange(epochs)

# Idealized loss trajectories (for illustration — real GAN losses are noisy and non-monotonic)
d_loss = 1.4 - 0.8 * (1 - np.exp(-t / 20)) + 0.05 * np.sin(t / 3)
g_loss = 2.0 * np.exp(-t / 25) + 0.6 + 0.08 * np.cos(t / 4)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(t, d_loss, label="Discriminator loss", color="steelblue", lw=2)
axes[0].plot(t, g_loss, label="Generator loss", color="darkorange", lw=2)
axes[0].set_title("Typical GAN Loss Curves (Illustrative)")
axes[0].set_xlabel("Training step")
axes[0].set_ylabel("Loss")
axes[0].legend()

# Toy alternating training: generator mean approaches real data mean
real_mean = 2.0
g_mu_hist = []
g_mu_train = 0.0
lr_g, lr_d = 0.15, 0.1
d_w_train, d_b_train = 0.3, -0.2

for step in range(epochs):
    # Update discriminator (push D(real) up, D(fake) down)
    fake = g_mu_train + np.random.randn(64)
    real = np.random.normal(real_mean, 0.3, 64)
    d_grad = (discriminate(real).mean() - 1) * real.mean() + discriminate(fake).mean() * fake.mean()
    d_w_train += lr_d * np.sign(d_grad) * 0.01

    # Update generator (move mean toward where D assigns high P(real))
    g_mu_train += lr_g * (real_mean - g_mu_train) * discriminate(np.array([g_mu_train]))[0]
    g_mu_hist.append(g_mu_train)

axes[1].plot(g_mu_hist, color="seagreen", lw=2)
axes[1].axhline(real_mean, color="gray", ls="--", label=f"Real mean ({real_mean})")
axes[1].set_title("Generator Mean Converging Toward Real Distribution")
axes[1].set_xlabel("Training step")
axes[1].set_ylabel("Generator mean")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Final generator mean: {g_mu_train:.3f} (target: {real_mean})")

---

## 6. GAN Applications

GANs have been applied across domains where generating realistic synthetic data provides value. While diffusion models now dominate high-fidelity image generation, GANs remain relevant in many production and research settings.

### 6.1 Image Generation and Editing

The most visible application. GANs generate photorealistic faces (StyleGAN), artwork, product images, and game assets. Latent space manipulation enables **semantic editing** — changing hair color, adding smiles, or aging faces without retraining. Tools like NVIDIA's StyleGAN demos illustrate this capability.

### 6.2 Image-to-Image Translation

CycleGAN and Pix2Pix translate images between domains: sketches to photos, day to night, horses to zebras, MRI to CT scans. Applications include creative tools, medical imaging, and satellite imagery enhancement.

### 6.3 Super-Resolution and Enhancement

**SRGAN** and related models upscale low-resolution images to high resolution with perceptually realistic detail. Used in video streaming, surveillance footage enhancement, and medical imaging where sensor resolution is limited.

### 6.4 Data Augmentation

When labeled data is scarce — rare diseases in medical imaging, minority classes in classification — GANs generate synthetic training examples. This improves model robustness without expensive data collection. Conditional GANs ensure generated samples carry correct labels.

### 6.5 Anomaly Detection

Train a GAN only on normal data. At inference, samples that the discriminator flags as fake or that the generator cannot reconstruct well are likely **anomalies**. Used in manufacturing defect detection, network intrusion detection, and medical screening.

### 6.6 Drug Discovery and Molecular Design

GANs generate molecular structures with desired properties — binding affinity, solubility, low toxicity. The generator proposes candidate molecules; a discriminator (or property predictor) filters for viability. This accelerates early-stage pharmaceutical research.

### 6.7 Deepfakes and Media Synthesis

GANs and their descendants enable face swapping, lip-syncing, and voice cloning. While these have legitimate uses in film production and accessibility, they also raise concerns about misinformation. Understanding GAN mechanics is essential for both creating and detecting synthetic media.

### 6.8 GANs in the Modern Generative AI Landscape

GANs laid the groundwork for adversarial and generative thinking in deep learning. Today, **diffusion models** (Stable Diffusion, DALL-E 3) and **autoregressive models** (GPT, image token models) produce higher-quality results for many tasks. However, GAN concepts persist:

- Adversarial training appears in model alignment and robustness research.
- StyleGAN's latent space manipulation informs controllable generation in newer architectures.
- CycleGAN's unpaired translation remains a standard approach where paired data is unavailable.
- GAN-based methods are still preferred when **fast inference** is required — a GAN generates in a single forward pass, while diffusion models require dozens to hundreds of iterative steps.

---

## 7. Summary

**Generative Adversarial Networks** learn to generate data by training a generator and discriminator in opposition. The generator maps random noise to synthetic samples; the discriminator distinguishes real from fake. Through this minimax game, the generator learns the data distribution implicitly — without explicit density modeling or reconstruction loss.

The core architecture pairs an upsampling generator with a downsampling discriminator. **Conditional GANs** add side information for controlled generation. **StyleGAN** achieves fine-grained control through style-based synthesis, and **CycleGAN** enables unpaired image translation via cycle consistency.

Training alternates between discriminator and generator updates using adversarial loss functions. Stability challenges — mode collapse, vanishing gradients, non-convergence — require careful architectural choices and techniques like label smoothing, spectral normalization, and Wasserstein objectives.

GANs power image generation, style transfer, super-resolution, data augmentation, anomaly detection, and molecular design. While diffusion and autoregressive models now lead many generation benchmarks, GANs remain important for fast inference, unpaired translation, and understanding the adversarial foundations of generative AI.